In [ ]:
# ============================================================
# CELL 1 - Environment Setup
# Installs the latest HuggingFace Transformers from source,
# required to support the Qwen3.5 model architecture (qwen3_5)
# which is not available in the Kaggle default transformers 5.0.0.
# Also upgrades huggingface_hub and datasets for compatibility.
# ============================================================
import subprocess, sys, importlib

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers @ git+https://github.com/huggingface/transformers.git"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub", "datasets"
])

mods = [k for k in sys.modules
        if any(x in k for x in ["transformers","huggingface_hub","datasets"])]
for m in mods: del sys.modules[m]

import site
for path in [site.getusersitepackages(),
             "/root/.local/lib/python3.12/site-packages"]:
    if path and path not in sys.path:
        sys.path.insert(0, path)

import transformers, huggingface_hub
importlib.reload(transformers)
importlib.reload(huggingface_hub)

print(f"Transformers: {transformers.__version__}")
print(f"HuggingFace Hub: {huggingface_hub.__version__}")

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("Qwen/Qwen3.5-0.8B", trust_remote_code=True)
print(f"Qwen3.5 recognized: {cfg.model_type} ✅")



In [ ]:
# ============================================================
# CELL 2 - Imports & Reproducibility
# Sets up all required libraries and fixes the random seed (42)
# across Python, NumPy, PyTorch and CUDA for full reproducibility.
# Defines global constants: label names, column names, fallback label.
# ============================================================
import os, random, re, gc, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm.auto import tqdm

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed: {SEED} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

LABELS         = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
LABEL2ID       = {l: i for i, l in enumerate(LABELS)}
FALLBACK_LABEL = "Ambivalent"
QUESTION_COL   = "question"
ANSWER_COL     = "interview_answer"
LABEL_COL      = "clarity_label"
print("Labels:", LABEL2ID)


In [ ]:
# ============================================================
# CELL 3 - Data Loading
# Loads the QEvasion (CLARITY) dataset from HuggingFace.
# Creates a stratified 80/20 train/validation split from the
# original train split since no validation split is provided.
# Reports class distribution across all three splits.
# ============================================================
dataset = load_dataset("ailsntua/QEvasion")
train_full_df = pd.DataFrame(dataset["train"])
test_df       = pd.DataFrame(dataset["test"])

train_df, val_df = train_test_split(
    train_full_df, test_size=0.2, random_state=SEED,
    stratify=train_full_df[LABEL_COL])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")
for name, df in [("Train",train_df),("Val",val_df),("Test",test_df)]:
    print(f"\n{name}:")
    for l in LABELS:
        n = (df[LABEL_COL]==l).sum()
        print(f"  {l:<20} {n} ({100*n/len(df):.1f}%)")

In [ ]:
# ============================================================
# CELL 4 - Length Feature Engineering
# Computes word-level question and answer lengths for each split.
# Creates binary subgroups (short/long) based on the training
# set median for later subgroup performance analysis.
# ============================================================
def add_length_features(df):
    df = df.copy()
    df["question_len"] = df[QUESTION_COL].fillna("").apply(lambda x: len(x.split()))
    df["answer_len"]   = df[ANSWER_COL].fillna("").apply(lambda x: len(x.split()))
    q_med = df["question_len"].median()
    a_med = df["answer_len"].median()
    df["question_group"] = df["question_len"].apply(
        lambda x: "short_q" if x <= q_med else "long_q")
    df["answer_group"] = df["answer_len"].apply(
        lambda x: "short_a" if x <= a_med else "long_a")
    return df, q_med, a_med

train_df, q_med, a_med = add_length_features(train_df)
val_df,   _,    _      = add_length_features(val_df)
test_df,  _,    _      = add_length_features(test_df)
print(f"Question median: {q_med} | Answer median: {a_med} words")

In [ ]:
# ============================================================
# CELL 5 - Class Distribution Visualization
# Plots and saves a bar chart of label counts across train,
# validation and test splits. Highlights the class imbalance
# (Ambivalent ~59-67%) which is the main challenge of this task.
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#4C72B0", "#DD8452", "#55A868"]
for ax, (name, df) in zip(axes, [("Train",train_df),
                                   ("Validation",val_df),
                                   ("Test",test_df)]):
    counts = df[LABEL_COL].value_counts().reindex(LABELS, fill_value=0)
    bars = ax.bar(LABELS, counts.values, color=colors)
    ax.set_title(f"{name} (n={len(df)})", fontweight="bold")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(0, counts.values.max()*1.2)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+5, str(v), ha="center", fontsize=10)
plt.suptitle("Class Distribution Across Splits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")

In [ ]:
# ============================================================
# CELL 6 - Submission Format Verification
# Loads the sample solution to confirm the required output format:
# two columns (Id, Predicted) with 308 rows matching the test set.
# ============================================================
sample_sub = pd.read_csv(
    "/kaggle/input/competitions/ys-19-2025-2026-assignment-3/sample_solution.csv")
print("Submission columns:", sample_sub.columns.tolist())
print("Shape:", sample_sub.shape)
print(sample_sub.head())

In [ ]:
# ============================================================
# CELL 7 - Few-shot Example Selection
# Selects 2 balanced examples per class (6 total) from the
# training set using fixed seed=42 for reproducibility.
# These examples are reused across all few-shot strategies.
# ============================================================
def select_fewshot_examples(df, n_per_class=2, seed=SEED):
    examples = []
    for label in LABELS:
        pool = df[df[LABEL_COL] == label]
        for _, row in pool.sample(n=n_per_class, random_state=seed).iterrows():
            examples.append({
                "question": row[QUESTION_COL],
                "answer":   row[ANSWER_COL],
                "label":    row[LABEL_COL]
            })
    return examples

fewshot_examples = select_fewshot_examples(train_df)
print(f"Selected {len(fewshot_examples)} few-shot examples:")
for i, ex in enumerate(fewshot_examples):
    print(f"  [{i+1}] {ex['label']}: {ex['question'][:70]}...")

In [ ]:
# ============================================================
# CELL 8 - Prompt Builders
# Implements all four prompting strategies as functions.
# Each follows the 4-component structure from the course slides:
# instruction + label definitions + input data + output indicator.
# System/user split is used as recommended in the TA's tutorial.
# - zero_shot:    label defs + Q/A + output constraint
# - few_shot:     6 labeled examples + Q/A + output constraint
# - cot:          step-by-step reasoning + Final Label: format
# - few_shot_cot: 3 examples with reasoning + same format
# ============================================================
LABEL_DEFS = (
    "- Clear Reply: The answer directly and clearly addresses the question.\n"
    "- Ambivalent: The answer partially addresses the question or is evasive/unclear.\n"
    "- Clear Non-Reply: The answer does not address the question at all."
)

def build_zeroshot_prompt(q, a):
    return (
        f"You are an expert in political discourse analysis.\n"
        f"Classify the clarity of a politician's answer to a journalist's question.\n\n"
        f"Label definitions:\n{LABEL_DEFS}\n\n"
        f"Question: {q}\n\nAnswer: {a}\n\n"
        f"Reply with exactly one of these labels and nothing else:\n"
        f"Clear Reply | Ambivalent | Clear Non-Reply"
    )

def build_fewshot_prompt(q, a, examples=None):
    if examples is None: examples = fewshot_examples
    demo_text = "Here are labeled examples:\n\n"
    for i, ex in enumerate(examples):
        demo_text += (
            f"Example {i+1}:\n"
            f"Question: {ex['question']}\n"
            f"Answer: {ex['answer'][:200]}...\n"
            f"Label: {ex['label']}\n\n"
        )
    return (
        f"You are an expert in political discourse analysis.\n"
        f"Classify the clarity of a politician's answer to a journalist's question.\n\n"
        f"Label definitions:\n{LABEL_DEFS}\n\n"
        f"{demo_text}"
        f"Now classify this new example:\n"
        f"Question: {q}\n\nAnswer: {a}\n\n"
        f"Reply with exactly one of these labels and nothing else:\n"
        f"Clear Reply | Ambivalent | Clear Non-Reply"
    )

def build_cot_prompt(q, a):
    return (
        f"You are an expert in political discourse analysis.\n\n"
        f"Label definitions:\n{LABEL_DEFS}\n\n"
        f"Question: {q}\n\nAnswer: {a}\n\n"
        f"Analyze in 2-3 sentences, then output the label.\n"
        f"Your response MUST end with this exact line:\n"
        f"Final Label: [Clear Reply or Ambivalent or Clear Non-Reply]\n\n"
        f"Analysis:"
    )

COT_REASONS = {
    "Clear Reply":    "The politician directly answers what was asked.",
    "Ambivalent":     "The politician partially addresses the question but avoids direct commitment.",
    "Clear Non-Reply":"The politician ignores the question entirely.",
}

def build_fewshot_cot_prompt(q, a, examples=None):
    if examples is None: examples = fewshot_examples
    demo_text = "Here are labeled examples with reasoning:\n\n"
    for i, ex in enumerate(examples[:3]):
        demo_text += (
            f"Example {i+1}:\n"
            f"Question: {ex['question']}\n"
            f"Answer: {ex['answer'][:150]}...\n"
            f"Reasoning: {COT_REASONS[ex['label']]}\n"
            f"Final Label: {ex['label']}\n\n"
        )
    return (
        f"You are an expert in political discourse analysis.\n"
        f"Classify the clarity of a politician's answer to a journalist's question.\n\n"
        f"Label definitions:\n{LABEL_DEFS}\n\n"
        f"{demo_text}"
        f"Now classify this new example using the same format:\n"
        f"Question: {q}\n\nAnswer: {a}\n\n"
        f"Reasoning: [your analysis]\n"
        f"Final Label: [Clear Reply | Ambivalent | Clear Non-Reply]"
    )

PROMPT_STRATEGIES = {
    "zero_shot":    lambda q, a: build_zeroshot_prompt(q, a),
    "few_shot":     lambda q, a: build_fewshot_prompt(q, a),
    "cot":          lambda q, a: build_cot_prompt(q, a),
    "few_shot_cot": lambda q, a: build_fewshot_cot_prompt(q, a),
}
print("Strategies:", list(PROMPT_STRATEGIES.keys()))

In [ ]:
# ============================================================
# CELL 9 - Output Parser
# Extracts the predicted label from raw model output using a
# cascaded matching strategy:
# 1. For CoT: regex extracts "Final Label: ..." line
# 2. Match last non-empty line against known label variants
# 3. Full text scan as fallback
# 4. Default to FALLBACK_LABEL ("Ambivalent") if nothing matches
# Also strips Qwen3.5 <think>...</think> blocks before parsing.
# ============================================================
LABEL_VARIANTS = {
    "Clear Reply": [
        "clear reply", "clearreply", "**clear reply**",
        "1. clear reply", "label: clear reply",
    ],
    "Ambivalent": [
        "ambivalent", "**ambivalent**",
        "label: ambivalent", "is ambivalent",
    ],
    "Clear Non-Reply": [
        "clear non-reply", "clear nonreply", "non-reply",
        "nonreply", "clear non reply",
        "label: clear non-reply", "label: clear non reply",
    ],
}

def strip_think_block(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL|re.IGNORECASE)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL|re.IGNORECASE)
    return text.strip()

def parse_label(text, strategy):
    if not text or not text.strip():
        return FALLBACK_LABEL, False
    text = strip_think_block(text)
    if not text.strip():
        return FALLBACK_LABEL, False

    if "cot" in strategy:
        m = re.search(r"final label\s*[:\-]\s*([^\n]+)", text, re.IGNORECASE)
        if m:
            candidate = m.group(1).strip().lower()
            for canonical in LABELS:
                if canonical.lower() in candidate:
                    return canonical, True
            for canonical, variants in LABEL_VARIANTS.items():
                for v in variants:
                    if v in candidate:
                        return canonical, True

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    last = lines[-1] if lines else text
    last_lower = last.lower()
    for canonical, variants in LABEL_VARIANTS.items():
        for v in variants:
            if v in last_lower:
                return canonical, True

    full_lower = text.lower()
    for canonical, variants in LABEL_VARIANTS.items():
        for v in variants:
            if v in full_lower:
                return canonical, True

    return FALLBACK_LABEL, False

print("=== Parser tests ===")
for txt, strat in [
    ("Clear Reply",                              "zero_shot"),
    ("Final Label: Clear Non-Reply",             "cot"),
    ("Reasoning: ...\nFinal Label: Ambivalent",  "few_shot_cot"),
    ("<think>thinking</think>\nAmbivalent",       "zero_shot"),
    ("",                                         "zero_shot"),
]:
    l, v = parse_label(txt, strat)
    print(f"  {repr(txt[:45]):<48} -> {l} (valid={v})")

In [ ]:
# ============================================================
# CELL 10 - Model Loading Functions
# Defines load_model() and unload_model() utilities.
# Models are loaded one at a time with float16 precision and
# device_map="auto" for automatic GPU placement.
# Thinking mode is disabled via enable_thinking=False in the
# chat template, producing clean direct label outputs.
# ============================================================
MODEL_NAMES = {
    "qwen3.5_0.8b": "Qwen/Qwen3.5-0.8B",
    "qwen3.5_2b":   "Qwen/Qwen3.5-2B",
    "qwen3.5_4b":   "Qwen/Qwen3.5-4B",
}

def load_model(model_key):
    name = MODEL_NAMES[model_key]
    print(f"\nLoading {model_key} ({name})...")
    tokenizer = AutoTokenizer.from_pretrained(
        name, trust_remote_code=True, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True)
    model.eval()
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"  VRAM: {used:.2f} / {total:.2f} GB")
    return tokenizer, model

def unload_model(model):
    del model; gc.collect(); torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated())/1e9
    print(f"  Unloaded. VRAM free: {free:.2f} GB")

print("Model functions ready.")

In [ ]:
# ============================================================
# CELL 11 - Inference & Evaluation Functions
# run_inference(): formats prompt using chat template with
#   system/user roles and enable_thinking=False, then generates
#   with greedy decoding (do_sample=False) for reproducibility.
# evaluate_model_strategy(): runs inference on all rows of a
#   dataframe for a given model/strategy and returns results.
# compute_metrics(): computes accuracy, per-class P/R/F1,
#   Macro F1, and invalid output rate.
# ============================================================

# Defined here as well to ensure availability across all cells
SYSTEM_MSG = (
    "You are an expert in political discourse analysis. "
    "Classify the clarity of a politician's answer to a journalist's question. "
    "Follow instructions exactly and output only the requested label."
)

def run_inference(prompt, tokenizer, model, strategy, max_new_tokens=100):
    messages = [{"role": "system", "content": SYSTEM_MSG},
                {"role": "user",   "content": prompt}]
    try:
        fmt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        fmt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(fmt, return_tensors="pt",
                       truncation=True, max_length=3072).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:],
                           skip_special_tokens=True)
    label, valid = parse_label(raw, strategy)
    return raw, label, valid

def evaluate_model_strategy(df, tokenizer, model, strategy_name, max_new_tokens=100):
    results, invalid = [], 0
    fn = PROMPT_STRATEGIES[strategy_name]
    for _, row in tqdm(df.iterrows(), total=len(df), desc=strategy_name):
        raw, label, valid = run_inference(
            fn(str(row[QUESTION_COL]), str(row[ANSWER_COL])),
            tokenizer, model, strategy_name, max_new_tokens)
        results.append({"raw_output":raw,"predicted_label":label,"is_valid":valid})
        if not valid: invalid += 1
    out = df.copy().reset_index(drop=True)
    for key in ["raw_output","predicted_label","is_valid"]:
        out[key] = [r[key] for r in results]
    print(f"  Invalid: {invalid}/{len(df)} ({100*invalid/len(df):.1f}%)")
    return out

def compute_metrics(results_df, model_key, strategy):
    y_true = results_df[LABEL_COL].tolist()
    y_pred = results_df["predicted_label"].tolist()
    acc = accuracy_score(y_true, y_pred)
    p,r,f1,_ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, labels=LABELS,
                        average="macro", zero_division=0)
    m = {"model":model_key,"strategy":strategy,
         "accuracy":round(acc,4),"macro_f1":round(macro_f1,4),
         "invalid_rate":round((~results_df["is_valid"]).mean(),4)}
    for i,label in enumerate(LABELS):
        s = label.replace(" ","_").lower()
        m[f"p_{s}"]=round(p[i],4)
        m[f"r_{s}"]=round(r[i],4)
        m[f"f1_{s}"]=round(f1[i],4)
    return m

def print_metrics(m):
    print(f"  Accuracy={m['accuracy']:.4f} | Macro F1={m['macro_f1']:.4f} | "
          f"Invalid={m['invalid_rate']*100:.1f}%")
    for l in LABELS:
        s=l.replace(" ","_").lower()
        print(f"    {l:<20} P={m[f'p_{s}']:.3f} R={m[f'r_{s}']:.3f} F1={m[f'f1_{s}']:.3f}")

print("All inference functions ready.")

In [ ]:
# ============================================================
# CELL 12 - Main Experiment Loop
# Evaluates all 12 combinations (3 models x 4 strategies) on
# the full validation set (690 examples).
# Models are loaded/unloaded individually to manage VRAM.
# Results are saved to CSV after each strategy for safety.
# CoT strategies use max_new_tokens=400 for reasoning space.
# ============================================================
eval_df = val_df.copy()
print(f"FULL MODE: {len(eval_df)} samples")

all_metrics     = []
all_results_dfs = {}

for model_key in MODEL_NAMES:
    print(f"\n{'='*60}\nMODEL: {model_key}\n{'='*60}")
    tokenizer, model = load_model(model_key)

    for strategy in PROMPT_STRATEGIES:
        set_seed(SEED)
        max_tok = 400 if "cot" in strategy else 100
        print(f"\n  Strategy: {strategy}")
        res_df = evaluate_model_strategy(
            eval_df, tokenizer, model, strategy, max_tok)
        m = compute_metrics(res_df, model_key, strategy)
        print_metrics(m)
        all_metrics.append(m)
        all_results_dfs[(model_key, strategy)] = res_df
        pd.DataFrame(all_metrics).to_csv(
            "/kaggle/working/metrics_intermediate.csv", index=False)

    unload_model(model)

# Save final metrics
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv("/kaggle/working/all_metrics.csv", index=False)
print("\nAll experiments complete.")
print(f"Best: {metrics_df.loc[metrics_df['macro_f1'].idxmax()]['model']} + "
      f"{metrics_df.loc[metrics_df['macro_f1'].idxmax()]['strategy']} | "
      f"Macro F1 = {metrics_df['macro_f1'].max():.4f}")

In [ ]:
# ============================================================
# CELL 13 - Results Analysis & Visualization
# Prints the full per-class metrics table for all 12 combinations.
# Generates 4 plots saved to /kaggle/working/:
#   - macro_f1_comparison.png: grouped bar chart by model/strategy
#   - model_size_effect.png:   line plot showing scaling effect
#   - heatmap_macro_f1.png:    heatmap of all combinations
#   - perclass_f1_4b.png:      per-class F1 for best model
# ============================================================
metrics_df = pd.DataFrame(all_metrics)

print("=" * 90)
print("COMPLETE RESULTS TABLE")
print("=" * 90)
print(f"{'Model':<15} {'Strategy':<15} {'Acc':>6} {'MacroF1':>8} "
      f"{'P_CR':>6} {'R_CR':>6} {'F1_CR':>6} "
      f"{'P_AM':>6} {'R_AM':>6} {'F1_AM':>6} "
      f"{'P_CNR':>6} {'R_CNR':>6} {'F1_CNR':>6}")
print("-" * 90)
for _, row in metrics_df.iterrows():
    print(f"{row['model']:<15} {row['strategy']:<15} "
          f"{row['accuracy']:>6.3f} {row['macro_f1']:>8.3f} "
          f"{row['p_clear_reply']:>6.3f} {row['r_clear_reply']:>6.3f} {row['f1_clear_reply']:>6.3f} "
          f"{row['p_ambivalent']:>6.3f} {row['r_ambivalent']:>6.3f} {row['f1_ambivalent']:>6.3f} "
          f"{row['p_clear_non-reply']:>6.3f} {row['r_clear_non-reply']:>6.3f} {row['f1_clear_non-reply']:>6.3f}")
print("=" * 90)
best = metrics_df.loc[metrics_df["macro_f1"].idxmax()]
print(f"\nBEST SYSTEM: {best['model']} + {best['strategy']}")
print(f"Validation Macro F1: {best['macro_f1']:.4f}")
print(f"Kaggle Test F1:      0.6229")
print(f"\nReason: The 4B model has the strongest instruction-following capability,")
print(f"correctly applying label definitions without demonstrations.")
print(f"Zero-shot avoids negative few-shot transfer and keeps prompts concise.")

# Plot 1: Grouped bar chart
models_list = list(metrics_df["model"].unique())
strategies  = list(PROMPT_STRATEGIES.keys())
bar_colors  = ["#4C72B0","#DD8452","#55A868","#C44E52"]
x = np.arange(len(models_list)); width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, strategy in enumerate(strategies):
    vals = [metrics_df[(metrics_df["model"]==m) &
                       (metrics_df["strategy"]==strategy)]["macro_f1"].values[0]
            for m in models_list]
    bars = ax.bar(x + i*width, vals, width, label=strategy, color=bar_colors[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{v:.3f}", ha="center", fontsize=7, rotation=90)
ax.set_xlabel("Model"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 by Model and Prompting Strategy", fontweight="bold")
ax.set_xticks(x + width*1.5); ax.set_xticklabels(["0.8B","2B","4B"])
ax.legend(loc="upper left"); ax.set_ylim(0, 0.65); ax.grid(axis="y", alpha=0.3)
ax.axhline(y=best["macro_f1"], color="red", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/kaggle/working/macro_f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 2: Model size effect
fig, ax = plt.subplots(figsize=(9, 4))
for i, strategy in enumerate(strategies):
    vals = [metrics_df[(metrics_df["model"]==m) &
                       (metrics_df["strategy"]==strategy)]["macro_f1"].values[0]
            for m in models_list]
    ax.plot(["0.8B","2B","4B"], vals, marker="o",
            label=strategy, color=bar_colors[i], linewidth=2, markersize=8)
ax.set_xlabel("Model Size"); ax.set_ylabel("Macro F1")
ax.set_title("Effect of Model Size on Macro F1", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.savefig("/kaggle/working/model_size_effect.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 3: Heatmap
pivot = metrics_df.pivot(index="model", columns="strategy", values="macro_f1")
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label":"Macro F1"})
ax.set_title("Macro F1: Model × Strategy", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/heatmap_macro_f1.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 4: Per-class F1 for best model
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, strategy in zip(axes, strategies):
    row = metrics_df[(metrics_df["model"]=="qwen3.5_4b") &
                     (metrics_df["strategy"]==strategy)].iloc[0]
    f1_vals = [row[f"f1_{l.replace(' ','_').lower()}"] for l in LABELS]
    bars = ax.bar(LABELS, f1_vals, color=bar_colors[:3])
    ax.set_title(f"{strategy}", fontweight="bold", fontsize=10)
    ax.set_ylabel("F1"); ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=20)
    for bar, v in zip(bars, f1_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f"{v:.2f}", ha="center", fontsize=9)
plt.suptitle("Per-class F1: Qwen3.5-4B (Best Model)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/perclass_f1_4b.png", dpi=150, bbox_inches="tight")
plt.show()
print("All plots saved.")

In [ ]:
# ============================================================
# CELL 14 - Subgroup Analysis
# Splits validation results by question length and answer length
# (short/long based on training set medians) for the best system.
# Reports Macro F1 per subgroup to show which instances are harder.
# ============================================================
print("=== SUBGROUP ANALYSIS: Qwen3.5-4B, zero-shot ===\n")
best_res = all_results_dfs[("qwen3.5_4b", "zero_shot")].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (group_col, group_name) in zip(axes, [
        ("question_group","Question Length"),
        ("answer_group","Answer Length")]):
    groups = sorted(best_res[group_col].unique())
    macro_f1s = []
    for g in groups:
        sub = best_res[best_res[group_col]==g]
        mf1 = f1_score(sub[LABEL_COL], sub["predicted_label"],
                       labels=LABELS, average="macro", zero_division=0)
        acc = accuracy_score(sub[LABEL_COL], sub["predicted_label"])
        macro_f1s.append(mf1)
        print(f"  {group_name} {g:<12} n={len(sub):>3} | "
              f"Acc={acc:.3f} | Macro F1={mf1:.3f}")
    ax.bar(groups, macro_f1s, color=["#4C72B0","#DD8452"])
    ax.set_title(f"Macro F1 by {group_name}\n(Qwen3.5-4B, zero_shot)",
                 fontweight="bold")
    ax.set_ylabel("Macro F1"); ax.set_ylim(0, 0.6)
    for i, v in enumerate(macro_f1s):
        ax.text(i, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")
    print()

plt.tight_layout()
plt.savefig("/kaggle/working/subgroup_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Subgroup analysis complete.")

In [ ]:
# ============================================================
# CELL 15 - Error Analysis
# Analyzes prediction errors for the best system (4B zero-shot).
# Reports error rate per class, most common misprediction pairs,
# and 3 concrete error examples with question, answer, and output.
# Generates and saves the confusion matrix.
# ============================================================
res = all_results_dfs[("qwen3.5_4b", "zero_shot")].copy()
errors  = res[res[LABEL_COL] != res["predicted_label"]]
correct = res[res[LABEL_COL] == res["predicted_label"]]

print(f"Total: {len(res)} | Correct: {len(correct)} | Errors: {len(errors)}")
print(f"Overall error rate: {len(errors)/len(res)*100:.1f}%\n")

# Confusion matrix
cm = confusion_matrix(res[LABEL_COL], res["predicted_label"], labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Qwen3.5-4B Zero-shot", fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== Error rate per class ===")
for label in LABELS:
    errs  = errors[errors[LABEL_COL]==label]
    total = res[res[LABEL_COL]==label]
    print(f"  {label:<20} {len(errs):>3}/{len(total):>3} "
          f"({100*len(errs)/max(len(total),1):.1f}% error rate)")

print("\n=== Most common misprediction pairs ===")
err_pairs = errors.groupby([LABEL_COL,"predicted_label"]).size().reset_index(name="count")
print(err_pairs.sort_values("count", ascending=False).to_string(index=False))

print("\n=== Concrete error examples ===")
shown = set()
for _, row in errors.iterrows():
    key = (row[LABEL_COL], row["predicted_label"])
    if key not in shown and len(shown) < 3:
        shown.add(key)
        print(f"\nTrue: {row[LABEL_COL]} → Predicted: {row['predicted_label']}")
        print(f"Q: {row[QUESTION_COL][:120]}")
        print(f"A: {row[ANSWER_COL][:200]}...")
        print(f"Raw output: {repr(row['raw_output'][:100])}")

In [ ]:
# ============================================================
# CELL 16 - Final Submission Generation
# Runs the best system (Qwen3.5-4B, zero-shot) on the full
# test set (308 examples) and saves predictions to CSV.
# Output format matches sample_solution.csv: Id + Predicted.
# This file is submitted to the Kaggle competition.
# ============================================================
BEST_MODEL    = "qwen3.5_4b"
BEST_STRATEGY = "zero_shot"

print(f"Generating test predictions...")
print(f"Best system: {BEST_MODEL} + {BEST_STRATEGY}")
print(f"Validation Macro F1: {best['macro_f1']:.4f}")
print(f"Test samples: {len(test_df)}\n")

tokenizer, model = load_model(BEST_MODEL)
set_seed(SEED)

test_results = evaluate_model_strategy(
    test_df, tokenizer, model, BEST_STRATEGY, max_new_tokens=100)

unload_model(model)

submission = pd.DataFrame({
    "Id":        range(len(test_results)),
    "Predicted": test_results["predicted_label"].values
})

print(f"\nPrediction distribution:")
print(submission["Predicted"].value_counts())

submission.to_csv("/kaggle/working/submission_best_prompting_system.csv", index=False)
print(f"\nSaved: submission_best_prompting_system.csv")

assert list(submission.columns) == ["Id","Predicted"], "Column mismatch!"
assert len(submission) == 308, f"Wrong row count: {len(submission)}"
print("Format verified ✅")
print(submission.head(10))